# Part 5 · Notebook 01 — Core indicators, done right

**Sessions:** S1 (Library architecture & helpers) · S2 (Core indicators) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. See how a registry turns scattered functions into a documented, testable library.
2. Write an EMA with TA-Lib's seeding, and see why pandas gives different early values.
3. Write Wilder's RSI smoothing and the True Range behind ATR.
4. Run the library contract (length, warm-up, no look-ahead) and catch a leaky indicator.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()

## 1. A registry, not a folder of functions

Every indicator in the library is registered with a decorator that records its **group** and its **look-back** (how many leading values are NaN). The catalog, the docs and the tests are all generated from this registry.

In [ ]:
@p.indicator("wma", "trend", lambda n=10: n - 1)
def wma(x, n=10):
    """Linearly weighted moving average. Look-back n−1."""
    w = np.arange(1, n + 1)
    return pd.Series(x).rolling(n).apply(lambda v: (v * w).sum() / w.sum(), raw=True).to_numpy()

pd.DataFrame([{"name": k, "group": v["group"], "look-back (defaults)": v["lookback"](), "doc": v["doc"]}
              for k, v in p.REGISTRY.items()])

## 2. The EMA and its seed

`EMA[t] = α·x[t] + (1 − α)·EMA[t−1]` with `α = 2/(n+1)`. The recursion needs a starting value, and **the seed is where libraries disagree**. TA-Lib (and our library) starts at bar `n−1` with the **simple average of the first `n` values**, and reports NaN before that (look-back `n−1`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def ema(x, n=20):
    x = np.asarray(x, dtype=float)
    out = np.full(x.shape, np.nan)
    if x.size < n:
        return out
    a = 2.0 / (n + 1)
    out[n - 1] = ...                              # ✍️ the seed: the mean of the first n values
    for i in range(n, x.size):
        out[i] = ...                              # ✍️ the EMA recursion
    return out

df = p.synthetic_ohlcv(1500, seed=7)
o, h, l, c, v = p.arrays(df)
mine = p.attempt(ema, c, 20)
mine = p.check("ema", mine, p.ema(c, 20))
mine[15:25].round(4)

`pandas.ewm(span=n, adjust=False)` seeds with the **first value** instead and has no NaN warm-up. The two agree eventually, because the seed's weight decays as `(1 − α)^t`, but not at the start. This is the cause of most "my RSI differs from TradingView" questions.

In [ ]:
gap = np.abs(p.ema(c, 20) - p.ema_pandas(c, 20))
fig, ax = plt.subplots()
ax.semilogy(np.arange(len(c)), gap)
ax.axhline(1e-8, color=p.PALETTE[7], ls="--", lw=1, label="golden-test tolerance 1e-8")
ax.set(xlim=(0, 300), xlabel="bar", ylabel="|TA-Lib seed − pandas seed|", title="EMA(20): the seeding gap decays, slowly")
ax.legend(); plt.show()
first_ok = int(np.argmax(gap < 1e-8))
print(f"the two conventions differ by more than 1e-8 until bar {first_ok}")

## 3. Wilder's RSI

RSI smooths average gains and losses with **Wilder's** method, which is an EMA with `α = 1/n`: `avg = (avg·(n−1) + new) / n`. The averages are seeded with the plain mean of the first `n` changes, so the first RSI is at bar `n`. `gain` and `loss` below hold the price changes (`np.diff`), so the change *into* bar `i` is at position `i − 1`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def rsi(close, n=14):
    out = np.full(close.shape, np.nan)
    d = np.diff(close)
    gain, loss = np.where(d > 0, d, 0.0), np.where(d < 0, -d, 0.0)
    ag, al = gain[:n].mean(), loss[:n].mean()
    out[n] = 100.0 if al == 0 else 100.0 - 100.0 / (1.0 + ag / al)
    for i in range(n + 1, close.size):
        ag = ...                                  # ✍️ Wilder-smooth the average gain with gain[i - 1]
        al = ...                                  # ✍️ and the average loss with loss[i - 1]
        out[i] = 100.0 if al == 0 else 100.0 - 100.0 / (1.0 + ag / al)
    return out

mine = p.attempt(rsi, c, 14)
mine = p.check("rsi", mine, p.rsi(c, 14))
mine[12:20].round(3)

## 4. True Range and ATR

The True Range includes the overnight gap: `max(H − L, |H − C₋₁|, |L − C₋₁|)`. Bar 0 has no previous close, so its TR is NaN. ATR is the Wilder average of TR (look-back `n`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def true_range(high, low, close):
    prev = np.concatenate([[np.nan], close[:-1]])
    tr = ...                                      # ✍️ the largest of the three distances, bar by bar
    tr[0] = np.nan
    return tr

mine = p.attempt(true_range, h, l, c)
mine = p.check("true_range", mine, p.true_range(h, l, c))
gap_share = np.nanmean(p.true_range(h, l, c) > (h - l) + 1e-12)
print(f"on {gap_share:.0%} of bars the gap makes the true range larger than the bar's own range")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(df.index, c, lw=1, label="close"); axes[0].plot(df.index, p.ema(c, 50), label="EMA 50"); axes[0].legend()
axes[1].plot(df.index, p.rsi(c, 14), lw=1, color=p.PALETTE[1]); axes[1].axhline(70, ls="--", lw=1); axes[1].axhline(30, ls="--", lw=1)
axes[1].set_ylabel("RSI 14")
axes[2].plot(df.index, p.atr(h, l, c, 14) / c * 100, lw=1, color=p.PALETTE[2]); axes[2].set_ylabel("ATR 14, % of price")
axes[0].set_title("Core indicators on synthetic bars (note the volatility clusters in ATR)")
plt.tight_layout(); plt.show()

## 5. The library contract

Every indicator must keep the **same length** as its input, be NaN **exactly** during its look-back, and have **no look-ahead**: values up to bar `t` must not change when the data after `t` is deleted. `p.audit` checks all three. Here it runs over the registry, and over a centered moving average that looks perfectly reasonable on a chart.

In [ ]:
rows = {name: p.audit(lambda x, f=spec["fn"]: f(x), c, spec["lookback"]()) for name, spec in p.REGISTRY.items() if name != "atr"}
rows["centered MA (leaky)"] = p.audit(p.leaky_smooth, c, 2)
pd.DataFrame(rows).T.replace({True: "✔", False: "✘"})

A centered average uses `n//2` future bars: every backtest that uses it is fiction. The truncation test catches it in one line.

## Wrap-up

* Register every indicator with its look-back; generate the catalog and the tests from the registry.
* Know your seeding convention and document it; golden tests compare from bar 0.
* Run the contract on everything: length, warm-up, truncation.
* Graded version: `labs/part05/week17_core` (10 indicators matched to TA-Lib at 1e-8 with golden files).